# 1. Business Question and Target Variable

## Primary Business Question
Can we accurately predict the SalePriceUSD of a property using its physical characteristics (size, age, condition), neighborhood context, and prevailing market conditions (mortgage rates) to       provide real estate agents with data-driven pricing benchmarks?
### Target Definition
  - **Target Variable**: SalePriceUSD
  - **Definition**: The final recorded transaction price of the property in U.S. Dollars.

### Why This Target Matters
Predicting the sale price is the core objective of real estate analytics. From a business perspective, an accurate prediction allows the agency to:

1) Establish competitive initial listing prices to minimize "time on market."

2) Identify properties that are currently underpriced or overpriced relative to their features.

3) Forecast potential revenue and agent commissions for better financial planning.

### What One Row Represents
In this dataset, a single row represents a unique property listing that has been sold, containing its specific attributes (e.g., SqFt, Beds, Baths), its location context (Neighborhood, DistanceToCityMiles), and the financial environment at the time of sale (MortgageRatePct).

### Columns to be Removed
To ensure the model generalizes well and adheres to best practices, the following columns will be dropped:
1) ListingID: This is a unique identifier/index for each row. It contains no predictive patterns and would lead to overfitting if included.
2) PostSaleAppraisalUSD: This column must be removed to prevent Data Leakage.

### Data Leakage Identification
- **Leakage Column**: PostSaleAppraisalUSD
- **Reason**: This value is determined after the sale has occurred. In a real-world predictive scenario, we would not know the post-sale appraisal value at the time we are trying to predict the listing price. Including it would artificially inflate the model's accuracy because the appraisal is directly informed by the final sale price.

In [20]:
import numpy as np
import pandas as pd
# Create plots/visualizations (e.g., histograms, scatter plots, residual plots)
import matplotlib.pyplot as plt

# Split data into train/test sets; optionally compute cross-validated scores
from sklearn.model_selection import train_test_split, cross_val_score  
# Apply different preprocessing steps to different column groups (numeric vs categorical)
from sklearn.compose import ColumnTransformer  
# Chain preprocessing + modeling steps into one reproducible workflow
from sklearn.pipeline import Pipeline  
# Encode categorical variables; standardize numeric features (mean 0, std 1)
from sklearn.preprocessing import OneHotEncoder, StandardScaler  
# Fill missing values using a chosen strategy (e.g., median, most_frequent)
from sklearn.impute import SimpleImputer
# Fit a Multiple Linear Regression model
from sklearn.linear_model import LinearRegression
# Evaluate regression predictions using MAE, MSE/RMSE, and R²
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 

In [35]:
path = "IST660_IA4_synthetic_real_estate.csv"

# Load the data
# keep_default_na=False ensures specific strings are not auto-converted, 
# allowing us to handle "blanks to NaN" manually as required by instructions.
df = pd.read_csv(path, keep_default_na=False)  

# Display basic structural information
print("Shape:", df.shape)
print("Shape (rows, cols):", df.shape)
print("\nColumns in dataset:", df.columns.tolist())
# Check data types and non-null counts
df.info()
display(df.head(5))
display(df.tail(5))

Shape: (900, 17)
Shape (rows, cols): (900, 17)

Columns in dataset: ['ListingID', 'Neighborhood', 'PropertyType', 'Condition', 'SqFt', 'LotSqFt', 'Beds', 'Baths', 'YearBuilt', 'DistanceToCityMiles', 'SchoolRating', 'CrimeIndex', 'MortgageRatePct', 'TotalRooms', 'OnlineEstimateUSD', 'SalePriceUSD', 'PostSaleAppraisalUSD']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ListingID             900 non-null    int64  
 1   Neighborhood          900 non-null    object 
 2   PropertyType          900 non-null    object 
 3   Condition             900 non-null    object 
 4   SqFt                  900 non-null    object 
 5   LotSqFt               900 non-null    object 
 6   Beds                  900 non-null    int64  
 7   Baths                 900 non-null    object 
 8   YearBuilt             900 non-null    int64  
 9   Distan

,ListingID,Neighborhood,PropertyType,Condition,SqFt,LotSqFt,Beds,Baths,YearBuilt,DistanceToCityMiles,SchoolRating,CrimeIndex,MortgageRatePct,TotalRooms,OnlineEstimateUSD,SalePriceUSD,PostSaleAppraisalUSD
0,500001,Rural,House,Good,2216,9938,3,2.5,1956,14.19,4.8,48.5,6.36,9.5,487927.0,490004.0,459168.0
1,500002,University,House,Good,3028,5609,4,2.0,2022,9.43,7.6,38.4,4.97,8.0,738356.0,786742.0,788502.0
2,500003,Rural,Townhome,Good,2035,5331,3,2.5,2008,8.75,6.0,33.5,7.35,7.5,477247.0,467777.0,483201.0
3,500004,Suburb,Condo,,2642,8452,2,2.5,1986,21.96,6.6,39.5,6.27,8.5,644423.0,667625.0,677707.0
4,500005,Suburb,Condo,Good,1182,13430,1,3.5,2011,0.30,6.8,36.6,5.96,6.5,466989.0,463497.0,473091.0


,ListingID,Neighborhood,PropertyType,Condition,SqFt,LotSqFt,Beds,Baths,YearBuilt,DistanceToCityMiles,SchoolRating,CrimeIndex,MortgageRatePct,TotalRooms,OnlineEstimateUSD,SalePriceUSD,PostSaleAppraisalUSD
895,500896,University,Condo,Good,1279,4344,5,1.0,1962,4.22,8.5,32.8,5.76,8.0,319714.0,361583.0,360041.0
896,500897,Rural,House,Fair,1092,2552,3,3.0,1976,12.05,7.3,54.8,5.80,8.0,237791.0,257075.0,230748.0
897,500898,Suburb,Townhome,Fair,1900,2229,4,2.0,1976,6.37,6.8,60.3,6.17,9.0,512403.0,543375.0,524858.0
898,500899,Rural,Condo,Good,1916,8898,5,1.0,2020,9.27,7.7,58.6,5.44,10.0,480664.0,500486.0,485861.0
899,500900,Downtown,House,Good,1295,3994,2,2.5,2009,11.15,8.1,16.6,5.93,9.5,553490.0,589057.0,592484.0
